# LAB DAY 19: XÂY DỰNG HỆ THỐNG GRAPHRAG

Notebook này hướng dẫn từng bước xây dựng hệ thống GraphRAG và so sánh với Flat RAG.

## 1. Setup và Import Libraries

In [ ]:
import os
import sys
import json
from dotenv import load_dotenv

# Add src to path
sys.path.append('src')

from entity_extraction import EntityExtractor
from graph_builder import NetworkXGraphBuilder
from graph_rag import GraphRAG
from flat_rag import FlatRAG

# Load environment variables
load_dotenv()

print("✅ Libraries imported successfully!")

## 2. Đọc và Khám phá Corpus

In [ ]:
# Đọc corpus
corpus_path = "data/tech_company_corpus.txt"

with open(corpus_path, 'r', encoding='utf-8') as f:
    corpus_text = f.read()

# Thống kê
paragraphs = [p.strip() for p in corpus_text.split('\n\n') if p.strip()]

print(f"📊 Thống kê Corpus:")
print(f"  - Tổng số ký tự: {len(corpus_text):,}")
print(f"  - Số đoạn văn: {len(paragraphs)}")
print(f"\n📝 Ví dụ đoạn văn đầu tiên:")
print(paragraphs[0])

## 3. Trích xuất Thực thể và Quan hệ (Entity & Relation Extraction)

In [ ]:
# Khởi tạo extractor
extractor = EntityExtractor()

# Test với một đoạn văn
test_text = paragraphs[0]
print(f"🔍 Test trích xuất từ đoạn văn:\n{test_text}\n")

test_triples = extractor.extract_triples(test_text)

print(f"\n✅ Đã trích xuất {len(test_triples)} triples:")
for s, p, o in test_triples:
    print(f"  ({s}) --[{p}]--> ({o})")

In [ ]:
# Trích xuất từ toàn bộ corpus
print("🚀 Bắt đầu trích xuất từ toàn bộ corpus...\n")

all_triples = extractor.extract_from_corpus(corpus_path)

print(f"\n✅ Tổng số triples: {len(all_triples)}")

# Khử trùng
unique_triples = extractor.deduplicate_triples(all_triples)
print(f"✅ Sau khử trùng: {len(unique_triples)} triples")

In [ ]:
# Hiển thị một số triples mẫu
print("\n📋 10 triples đầu tiên:")
for i, (s, p, o) in enumerate(unique_triples[:10], 1):
    print(f"{i:2d}. ({s}) --[{p}]--> ({o})")

## 4. Xây dựng Knowledge Graph

In [ ]:
# Khởi tạo graph builder
graph_builder = NetworkXGraphBuilder()

# Thêm triples vào đồ thị
print("🏗️  Đang xây dựng đồ thị...")
graph_builder.add_triples(unique_triples)

# Thống kê
stats = graph_builder.get_stats()
print(f"\n📊 Thống kê Knowledge Graph:")
print(f"  - Số nodes (thực thể): {stats['num_nodes']}")
print(f"  - Số edges (quan hệ): {stats['num_edges']}")
print(f"  - Mật độ đồ thị: {stats['density']:.4f}")
print(f"  - Liên thông: {'Có' if stats['is_connected'] else 'Không'}")

In [ ]:
# Trực quan hóa đồ thị
print("🎨 Đang tạo visualization...")
graph_builder.visualize("output/graph_visualization.png", figsize=(24, 18))
print("✅ Đã lưu visualization vào output/graph_visualization.png")

# Hiển thị trong notebook (nếu có)
from IPython.display import Image, display
display(Image(filename="output/graph_visualization.png"))

## 5. Test Graph Traversal (Duyệt đồ thị)

In [ ]:
# Test duyệt đồ thị từ một thực thể
test_entity = "OpenAI"
max_hops = 2

print(f"🔍 Duyệt đồ thị từ '{test_entity}' trong phạm vi {max_hops} hops:\n")

neighbors = graph_builder.get_neighbors(test_entity, max_hops=max_hops)

print(f"✅ Tìm thấy {len(neighbors)} triples liên quan:")
for i, (s, p, o) in enumerate(neighbors[:20], 1):  # Hiển thị 20 triples đầu
    print(f"{i:2d}. ({s}) --[{p}]--> ({o})")

if len(neighbors) > 20:
    print(f"\n... và {len(neighbors) - 20} triples khác")

## 6. Thiết lập GraphRAG

In [ ]:
# Khởi tạo GraphRAG
graph_rag = GraphRAG(graph_builder)

print("✅ GraphRAG đã sẵn sàng!")

In [ ]:
# Test với một câu hỏi
test_question = "OpenAI được thành lập bởi ai và vào năm nào?"

print(f"❓ Câu hỏi: {test_question}\n")

result = graph_rag.answer_query(test_question, max_hops=2, verbose=True)

print(f"\n📝 Metadata:")
print(f"  - Entities: {result['entities']}")
print(f"  - Số facts: {result['num_facts']}")
print(f"  - Max hops: {result['max_hops']}")

## 7. Thiết lập Flat RAG (Baseline)

In [ ]:
# Khởi tạo Flat RAG
flat_rag = FlatRAG()

# Index corpus
print("📚 Đang index corpus vào ChromaDB...")
flat_rag.index_corpus(corpus_path)

print("✅ Flat RAG đã sẵn sàng!")

In [ ]:
# Test với cùng câu hỏi
print(f"❓ Câu hỏi: {test_question}\n")

flat_result = flat_rag.answer_query(test_question, top_k=3, verbose=True)

print(f"\n📝 Metadata:")
print(f"  - Số documents: {flat_result['num_docs']}")

## 8. So sánh GraphRAG vs Flat RAG

In [ ]:
# Danh sách câu hỏi test
test_questions = [
    "OpenAI được thành lập bởi ai và vào năm nào?",
    "Sam Altman có vai trò gì tại OpenAI?",
    "Elon Musk có liên quan gì đến OpenAI và Tesla?",
    "Microsoft đã đầu tư bao nhiêu vào OpenAI?",
    "Những công ty nào được thành lập bởi Elon Musk?",
]

print(f"🧪 Đánh giá trên {len(test_questions)} câu hỏi\n")
print("="*100)

In [ ]:
# So sánh từng câu hỏi
comparison_results = []

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*100}")
    print(f"Câu hỏi {i}/{len(test_questions)}: {question}")
    print(f"{'='*100}")
    
    # GraphRAG
    print("\n[GraphRAG]")
    graph_result = graph_rag.answer_query(question, max_hops=2, verbose=False)
    print(f"Answer: {graph_result['answer']}")
    print(f"Facts used: {graph_result['num_facts']}")
    
    # Flat RAG
    print("\n[Flat RAG]")
    flat_result = flat_rag.answer_query(question, top_k=3, verbose=False)
    print(f"Answer: {flat_result['answer']}")
    print(f"Docs used: {flat_result['num_docs']}")
    
    comparison_results.append({
        "question": question,
        "graph_rag": graph_result,
        "flat_rag": flat_result
    })

## 9. Phân tích Kết quả

In [ ]:
# Tạo bảng so sánh
import pandas as pd

comparison_df = pd.DataFrame([
    {
        "Câu hỏi": r["question"][:50] + "...",
        "GraphRAG Answer": r["graph_rag"]["answer"][:100] + "...",
        "Flat RAG Answer": r["flat_rag"]["answer"][:100] + "...",
        "Graph Facts": r["graph_rag"]["num_facts"],
        "Flat Docs": r["flat_rag"]["num_docs"]
    }
    for r in comparison_results
])

print("\n📊 Bảng so sánh kết quả:\n")
print(comparison_df.to_string(index=False))

## 10. Lưu Kết quả

In [ ]:
# Lưu triples
os.makedirs("output", exist_ok=True)

with open("output/triples.json", 'w', encoding='utf-8') as f:
    json.dump(
        [{"subject": s, "predicate": p, "object": o} for s, p, o in unique_triples],
        f, ensure_ascii=False, indent=2
    )

# Lưu đồ thị
graph_builder.save("output/knowledge_graph.json")

# Lưu kết quả so sánh
with open("output/comparison_results.json", 'w', encoding='utf-8') as f:
    json.dump(comparison_results, f, ensure_ascii=False, indent=2)

print("✅ Đã lưu tất cả kết quả vào thư mục output/")
print("  - triples.json")
print("  - knowledge_graph.json")
print("  - graph_visualization.png")
print("  - comparison_results.json")

## 11. Kết luận

### Ưu điểm của GraphRAG:
1. **Multi-hop reasoning**: Có thể kết nối thông tin qua nhiều bước
2. **Structured knowledge**: Tri thức được tổ chức có cấu trúc rõ ràng
3. **Explainable**: Có thể trace được nguồn gốc thông tin

### Nhược điểm:
1. **Chi phí cao hơn**: Cần nhiều API calls để extract triples
2. **Phụ thuộc vào extraction quality**: Nếu extract sai, toàn bộ hệ thống bị ảnh hưởng
3. **Phức tạp hơn**: Cần maintain đồ thị

### Khi nào nên dùng GraphRAG:
- Dữ liệu có nhiều mối quan hệ phức tạp
- Cần trả lời câu hỏi multi-hop
- Cần explainability cao
- Domain knowledge rõ ràng (entities, relations)